# Assignment — Azure OpenAI + Azure AI Search RAG

**Program:** Brilian Sistem Informasi Bootcamp  
**Session 35:** Azure OpenAI and AI Search

## 🎯 Learning Objectives

By completing this assignment, you will:
1. Design an Azure AI Search **index schema with multiple field types** (searchable, filterable, sortable, facetable)
2. Configure and use **semantic ranking** to improve retrieval ordering
3. Implement **category filtering** alongside free-text search
4. Build a **multi-turn RAG chatbot** that maintains conversation history
5. **Compare keyword search vs semantic search** on the same corpus and discuss the difference

## 🚗 Use Case — Mitsubishi Vehicle Service Manual Assistant

You are the AI engineer at BSI. The After-Sales division of Mitsubishi Indonesia wants an internal assistant that helps service advisors quickly answer customer questions about common vehicle issues — engine warning lights, battery problems, transmission service, A/C maintenance, etc. Your assistant will retrieve from a curated technical knowledge base and ground the LLM's answer on it, with category filtering and citations.

The corpus is richer than the hands-on: each article has a title, a description, a solution, a category (engine / battery / transmission / brakes / hvac / electrical / fuel / suspension / cooling / general), a severity level (low / medium / high), a model compatibility list, and a last-updated date. Your index design has to make all of these usable.

## 📅 Submission

| Item | Detail |
|---|---|
| Platform | Google Classroom |
| File naming | `NamaLengkap_Assignment_AzureOpenAI_AISearch.ipynb` |
| Deadline | _[insert deadline]_ |
| Late policy | _[insert policy]_ |

## ⚙️ Setup

> ⚠️ **IMPORTANT — Never hardcode your API keys.** Always use Google Colab Secrets.

**How to set it up:**
1. Click the 🔑 icon on the left sidebar in Colab
2. Add the following secrets and toggle **Notebook access** ON for each:
   - `AZURE_OPENAI_API_KEY`
   - `AZURE_OPENAI_ENDPOINT`
   - `AZURE_OPENAI_DEPLOYMENT`
   - `AZURE_SEARCH_API_KEY`
   - `AZURE_SEARCH_ENDPOINT`

## 📋 Tasks Overview

You will work through 7 tasks. Imports and sample data are provided; the logic is for you to write.

| # | Task | What you must implement |
|---|---|---|
| 1 | Install & load credentials | (provided — just run) |
| 2 | Connect to Azure OpenAI | client object + smoke test |
| 3 | Design and create the index | schema + semantic configuration |
| 4 | Upload the service manual corpus | (data provided — push it correctly) |
| 5 | Implement three retrieval modes | keyword, semantic, filtered |
| 6 | Build the multi-turn RAG chat | maintain message history |
| 7 | Evaluate & compare | run 5 questions, write your analysis |


## Task 1 — Install dependencies and load credentials

This part is provided. Just run it.

In [ ]:
!pip install -q openai==1.51.0 azure-search-documents==11.5.1 azure-core==1.30.2

In [ ]:
from google.colab import userdata

AZURE_OPENAI_API_KEY    = userdata.get("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT   = userdata.get("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = userdata.get("AZURE_OPENAI_DEPLOYMENT")

AZURE_SEARCH_API_KEY  = userdata.get("AZURE_SEARCH_API_KEY")
AZURE_SEARCH_ENDPOINT = userdata.get("AZURE_SEARCH_ENDPOINT")

for name, value in {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
    "AZURE_SEARCH_API_KEY": AZURE_SEARCH_API_KEY,
    "AZURE_SEARCH_ENDPOINT": AZURE_SEARCH_ENDPOINT,
}.items():
    print(f"{name:30s} -> {'OK' if value else 'MISSING'}")

## Task 2 — Connect to Azure OpenAI

**Your task:** create the `AzureOpenAI` client and verify the connection works.

**Hints (from the lecture):**
- The deployment name is what your code calls — not `gpt-4o`
- Use `api_version="2024-08-01-preview"` (or the version your resource exposes)
- Send a single short prompt and print the response to confirm

In [ ]:
# Imports are provided
from openai import AzureOpenAI

# TODO 2.1: instantiate the AzureOpenAI client using the credentials loaded above.
# aoai_client = AzureOpenAI(...)
aoai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2026-03-01"
)

# TODO 2.2: send a small test chat completion (e.g. "Reply with the word READY") and print it.
# Confirm you can see "READY" or similar before moving on.
response = aoai_client.chat.completions.create(
    model = AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {"role": "user", "content": "Reply with the word READY"}
    ],
    temperature=0,
    max_tokens=10.
)


## Task 3 — Design and create the AI Search index

**Your task:** create an index named `mitsubishi-service-manual` with the following schema, and configure **semantic search** on it.

| Field | Type | Required attributes |
|---|---|---|
| `id` | String | key |
| `title` | String | searchable |
| `problem_description` | String | searchable |
| `solution` | String | searchable |
| `category` | String | filterable, facetable |
| `severity` | String | filterable, facetable |
| `model_compatibility` | Collection(String) | filterable, facetable |
| `last_updated` | DateTimeOffset | filterable, sortable |

**Semantic configuration requirements:**
- Configuration name: `default-semantic-config`
- `title_field` = `title`
- `content_fields` = `problem_description`, `solution`
- `keywords_fields` = `category`

**Hints:**
- Look up `SearchableField`, `SimpleField`, `SimpleField(collection=True, ...)` and `SearchField` in the `azure.search.documents.indexes.models` module
- Semantic configuration uses `SemanticSearch`, `SemanticConfiguration`, `SemanticPrioritizedFields`, `SemanticField`
- If an index with the same name already exists, delete it before recreating (this assignment iterates often)

In [ ]:
# Imports are provided
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

INDEX_NAME = "mitsubishi-service-manual"

# TODO 3.1: build a SearchIndexClient pointed at your AI Search resource.
# index_client = SearchIndexClient(...)
index_client = SearchIndexClient(
    endpoint=AZURE_OPENAI_ENDPOINT,
    credential=AzureKeyCredential(AZURE_OPENAI_API_KEY)
)


# TODO 3.2: define the eight fields described in the table above.
# Tip: model_compatibility is a Collection(String) — use SearchField with collection=True
# fields = [ ... ]
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="problem_description", type=SearchFieldDataType.String),
    SearchableField(name="solution", type=SearchFieldDataType.String),
    SimpleField(
        name="category",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    SimpleField(
        name="severity",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    SearchField(
        name="model_compatibility",
        type=SearchFieldDataType.Collection(SearchFieldDataType.String),
        filterable=True,
        facetable=True,
    ),
    SimpleField(
        name="last_updated",
        type=SearchFieldDataType.DateTimeOffset,
        filterable=True,
        sortable=True,
    ),
]

# TODO 3.3: define the semantic configuration named "default-semantic-config"
# semantic_config = SemanticConfiguration(name=..., prioritized_fields=SemanticPrioritizedFields(...))
# semantic_search = SemanticSearch(configurations=[semantic_config])
semantic_config = SemanticConfiguration(
    name="default-semantic-config",
    prioritized_fields = SemanticPrioritizedFields(
        title_field = SemanticField(field_name="title"),
        content_fields=[
            SemanticField(field_name="problem_description"),
            SemanticField(field_name="solution")
        ],
        keywords_fields=[SemanticField(field_name="category")]
    )
)
semantic_search = SemanticSearch(configurations=[semantic_config])

# TODO 3.4: build the SearchIndex with fields=fields and semantic_search=semantic_search,
#           delete the existing index if present, then create it.
index = SearchIndex(name=INDEX_NAME, fields=fields, semantic_search=semantic_search)
try: 
    index_client.delete_index(INDEX_NAME)
except Exception: 
    pass
index_client.create_index(index)
print(f"Index '{INDEX_NAME}' created.")


## Task 4 — Upload the service manual corpus

The corpus is provided below (15 articles). **Your task** is to push it into the index using the `SearchClient`.

> Note: `last_updated` must be in **ISO-8601 format with timezone**, e.g. `"2024-09-12T00:00:00Z"`. The data is already formatted that way.

In [ ]:
service_manual_docs = [
    {
        "id": "svc-001",
        "title": "Engine check warning light is on",
        "problem_description": "The engine check warning light (yellow engine icon) appears on the dashboard. The engine may run normally or feel slightly rough. Common triggers include a loose fuel cap, faulty oxygen sensor, catalytic converter issue, or mass airflow sensor fault.",
        "solution": "First, tighten the fuel cap and drive for several cycles to see if the light clears. If not, scan the OBD-II port with a diagnostic tool to read the trouble code. Most common codes are P0420 (catalyst efficiency) and P0171 (lean mixture). Replace the indicated sensor if the code persists after clearing.",
        "category": "engine",
        "severity": "medium",
        "model_compatibility": ["Xpander", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-09-12T00:00:00Z",
    },
    {
        "id": "svc-002",
        "title": "Battery does not hold charge overnight",
        "problem_description": "Vehicle fails to start in the morning even though it was running fine the previous day. Headlights may appear dim. The battery voltage drops below 12.0V at rest. Often caused by parasitic draw, an aging battery, or a faulty alternator.",
        "solution": "Measure resting battery voltage with a multimeter. If below 12.4V, charge it fully and re-test after 12 hours. If voltage drops again, perform a parasitic draw test (should be under 50mA with all systems off). Replace the battery if it is older than 3 years and fails the load test. Check alternator output (should be 13.8-14.4V running).",
        "category": "battery",
        "severity": "high",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton"],
        "last_updated": "2024-10-03T00:00:00Z",
    },
    {
        "id": "svc-003",
        "title": "Automatic transmission slipping under acceleration",
        "problem_description": "Under hard acceleration the engine RPM rises but the vehicle does not accelerate proportionally. May be accompanied by delayed engagement when shifting from Park to Drive. Often indicates low or degraded transmission fluid.",
        "solution": "Check ATF level with the engine warm and the selector cycled through all gears. Inspect fluid color: bright red is healthy, brown indicates burnt fluid. Replace ATF and filter if degraded. If slipping persists after fluid service, internal clutch wear is likely and the transmission must be inspected by a certified Mitsubishi technician.",
        "category": "transmission",
        "severity": "high",
        "model_compatibility": ["Pajero Sport", "Outlander"],
        "last_updated": "2024-08-21T00:00:00Z",
    },
    {
        "id": "svc-004",
        "title": "Air conditioning blows warm air",
        "problem_description": "Cabin A/C blows ambient or warm air instead of cold air. May start cold and gradually warm up, or never cool at all. Most common cause is low refrigerant due to a leak; less commonly, a failed compressor clutch or blocked condenser.",
        "solution": "Connect manifold gauges to inspect low and high side pressures. If both are low, the system is undercharged: locate the leak with UV dye or electronic detector before recharging. If high side is very high, condenser airflow may be blocked - clean the condenser fins. If the compressor clutch is not engaging, check the clutch relay and pressure switches.",
        "category": "hvac",
        "severity": "medium",
        "model_compatibility": ["Xpander", "Xforce", "Outlander", "Pajero Sport", "Triton"],
        "last_updated": "2024-11-05T00:00:00Z",
    },
    {
        "id": "svc-005",
        "title": "Brake pedal feels soft or spongy",
        "problem_description": "Brake pedal travels further than normal before braking force is felt, or sinks slowly under continuous pressure. Vehicle may take longer to stop. Indicates air in the hydraulic system, a fluid leak, or a failing master cylinder.",
        "solution": "Inspect brake fluid level in the reservoir and look for external leaks at calipers, wheel cylinders, hoses, and the master cylinder. Bleed each caliper in the manufacturer-specified order (usually furthest from master cylinder first). If the pedal still sinks after bleeding, replace the master cylinder. Always use DOT 3 or DOT 4 fluid as specified in the owner manual.",
        "category": "brakes",
        "severity": "high",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-07-18T00:00:00Z",
    },
    {
        "id": "svc-006",
        "title": "Recommended engine oil change interval",
        "problem_description": "Customers frequently ask how often the engine oil should be changed and which oil grade Mitsubishi recommends.",
        "solution": "Standard recommendation is every 10,000 km or 6 months, whichever comes first, for normal driving conditions. For severe usage (heavy traffic, mountainous terrain, frequent short trips) shorten the interval to 5,000 km. Use the API SN/SP grade with viscosity 0W-20 or 5W-30 as specified on the engine oil cap.",
        "category": "engine",
        "severity": "low",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-12-01T00:00:00Z",
    },
    {
        "id": "svc-007",
        "title": "Power window does not respond from any switch",
        "problem_description": "One specific power window does not move up or down regardless of which switch is used (driver master switch or door switch). Other windows work normally. Usually a failed window motor or a broken regulator.",
        "solution": "Remove the door card and inspect the regulator for broken cables or plastic clips. Test the motor with 12V applied directly: if it runs, the regulator is the issue; if it does not, replace the motor assembly. Re-initialize the auto up/down function after reconnection per the workshop manual.",
        "category": "electrical",
        "severity": "low",
        "model_compatibility": ["Xpander", "Outlander"],
        "last_updated": "2024-06-30T00:00:00Z",
    },
    {
        "id": "svc-008",
        "title": "Engine overheating during heavy traffic",
        "problem_description": "Coolant temperature gauge climbs into the red zone in slow traffic but normalizes at highway speed. Steam may rise from under the hood. Indicates failed radiator fan, low coolant, or thermostat stuck closed.",
        "solution": "Stop driving immediately and let the engine cool. Inspect coolant level in the reservoir; top up with the recommended Mitsubishi long-life coolant if low. Verify the radiator fan engages when the engine is hot (around 95 C). If the fan fails, test the relay, fuse, and motor in sequence. Replace the thermostat if it does not open at the rated temperature when bench tested.",
        "category": "cooling",
        "severity": "high",
        "model_compatibility": ["Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-10-22T00:00:00Z",
    },
    {
        "id": "svc-009",
        "title": "Fuel consumption higher than expected",
        "problem_description": "Customer reports significant drop in fuel economy compared to manufacturer-stated figures. Often caused by underinflated tires, a clogged air filter, faulty oxygen sensor, or aggressive driving habits.",
        "solution": "Inspect tire pressure and set to the placard specification (typically 32-33 psi cold). Replace air filter if visibly dirty. Run an OBD-II scan to check for fuel system codes (P0171/P0172). Educate the customer on fuel-efficient driving habits: smooth acceleration, anticipating traffic, and avoiding excessive idling.",
        "category": "fuel",
        "severity": "low",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-09-28T00:00:00Z",
    },
    {
        "id": "svc-010",
        "title": "Knocking noise from front suspension over bumps",
        "problem_description": "A clunk or knock is heard from the front end when driving over bumps, speed bumps, or uneven roads. Steering may feel slightly loose. Common causes are worn stabilizer bar links, ball joints, or strut mounts.",
        "solution": "Lift the front of the vehicle and inspect each suspension component. Check stabilizer bar links by hand for play. Inspect ball joints with a pry bar - any vertical play indicates wear. Strut mounts should rotate smoothly without notchiness. Replace the worn component(s) and perform a wheel alignment afterwards.",
        "category": "suspension",
        "severity": "medium",
        "model_compatibility": ["Xpander", "Pajero Sport", "Triton"],
        "last_updated": "2024-08-09T00:00:00Z",
    },
    {
        "id": "svc-011",
        "title": "ABS warning light remains on after starting",
        "problem_description": "The ABS warning light remains illuminated after engine start. Standard brakes still function but the anti-lock feature is disabled. Most often a faulty wheel speed sensor or a damaged sensor harness near the wheel hub.",
        "solution": "Read ABS-specific trouble codes with a capable scan tool (a basic OBD-II reader is often insufficient). Codes typically point to a specific wheel sensor (FL, FR, RL, RR). Inspect the sensor and tone ring for debris and the wiring for damage. Clean or replace as needed. Clear the code and road test.",
        "category": "brakes",
        "severity": "high",
        "model_compatibility": ["Pajero Sport", "Outlander", "Triton"],
        "last_updated": "2024-11-19T00:00:00Z",
    },
    {
        "id": "svc-012",
        "title": "Coolant leaking from under the vehicle",
        "problem_description": "Green, pink, or orange fluid pooling under the vehicle after parking. Coolant level in the reservoir gradually drops. Common leak points include radiator seams, hose clamps, water pump weep hole, and heater core.",
        "solution": "Pressure-test the cooling system at 1.0 bar to identify the leak source. Inspect the radiator carefully along the plastic-to-metal seams. Check hose clamps for tightness and replace any swollen hoses. If the water pump weep hole is wet, replace the pump and the timing belt if applicable. Always refill with the correct Mitsubishi long-life coolant.",
        "category": "cooling",
        "severity": "high",
        "model_compatibility": ["Xpander", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-09-04T00:00:00Z",
    },
    {
        "id": "svc-013",
        "title": "Recommended tire pressure for daily driving",
        "problem_description": "Customers ask the correct tire pressure for daily driving and for full-load conditions.",
        "solution": "Use the pressure values printed on the door jamb placard, not the maximum value on the tire sidewall. For most Mitsubishi Indonesia models, daily driving is 32-33 psi front and rear, and 36 psi when fully loaded. Always check pressure cold (before driving). Add 2-3 psi if the customer regularly carries heavy loads.",
        "category": "general",
        "severity": "low",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-12-10T00:00:00Z",
    },
    {
        "id": "svc-014",
        "title": "Headlights flicker at idle",
        "problem_description": "Headlights pulse or dim noticeably at idle and stabilize when RPM rises. Indicates marginal alternator output, a weak battery, or loose ground straps.",
        "solution": "Measure system voltage at the battery with the engine running. It should be steady at 13.8-14.4V. If it drops below 13.5V at idle, perform an alternator load test. Inspect and tighten ground straps from the engine and chassis to the battery negative. Replace the battery if it fails the load test.",
        "category": "electrical",
        "severity": "medium",
        "model_compatibility": ["Pajero Sport", "Triton", "Outlander"],
        "last_updated": "2024-07-25T00:00:00Z",
    },
    {
        "id": "svc-015",
        "title": "Hard starting in the morning",
        "problem_description": "Engine cranks longer than normal before starting on the first cold start of the day, but starts immediately when warm. Often a weak battery, fuel pressure leak-down, or a worn starter motor.",
        "solution": "Test the battery cold cranking amps and replace if marginal. Check fuel rail pressure: it should hold pressure for at least 30 minutes after engine off. If it drops quickly, suspect leaking injectors or a failing fuel pump check valve. If cranking is slow and groany, the starter motor draw is high - test starter current and replace if out of spec.",
        "category": "engine",
        "severity": "medium",
        "model_compatibility": ["Xpander", "Xforce", "Pajero Sport", "Triton"],
        "last_updated": "2024-10-15T00:00:00Z",
    },
]

print(f"Loaded {len(service_manual_docs)} service articles.")
print(f"Categories: {sorted(set(d['category'] for d in service_manual_docs))}")
print(f"Severities: {sorted(set(d['severity'] for d in service_manual_docs))}")

In [ ]:
# Imports are provided
from azure.search.documents import SearchClient
import time

# TODO 4.1: build a SearchClient pointed at the index you just created.
# search_client = SearchClient(...)
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_API_KEY)
)

# TODO 4.2: upload service_manual_docs using the push API and report success count.
# result = search_client.upload_documents(documents=service_manual_docs)
# print(...)
result = search_client.upload_documents(documents=service_manual_docs)
succeeded = sum(1 for r in result if r.succeed)
print(f"Uploaded: {succeeded}/{len(service_manual_docs)} documents succeeded.")

# TODO 4.3: wait a few seconds and verify the document count matches what you uploaded.
# import time; time.sleep(3)
# print(...)
time.sleep(3)
count = search_client.get_document_count()
print(f"Index document count: {count} (expected {len(service_manual_docs)})")

## Task 5 — Implement three retrieval modes

You need three retrieval functions:

1. **`keyword_search(query, top_k)`** — plain BM25-style search using `search_text`
2. **`semantic_search(query, top_k)`** — uses `query_type=QueryType.SEMANTIC` and `semantic_configuration_name="default-semantic-config"`
3. **`filtered_search(query, category, top_k)`** — same as semantic but with an OData filter `category eq '<value>'`

Each function should return a list of dicts with the fields you care about (`id`, `title`, `problem_description`, `solution`, `category`, `severity`, and the score).

**Hints:**
- `QueryType.SEMANTIC` is in `azure.search.documents.models`
- For semantic search, also pass `query_caption` and `query_answer` if you want to see captions/answers in the result
- OData strings are case-sensitive: `category eq 'engine'` — single quotes around the value
- Look at `r['@search.score']` and (for semantic) `r['@search.reranker_score']`

In [ ]:
# Imports are provided
from azure.search.documents.models import QueryType, QueryCaptionType, QueryAnswerType


def keyword_search(query: str, top_k: int = 5) -> list[dict]:
    """Plain keyword (BM25) search."""
    # TODO 5.1: call search_client.search(...) with no query_type override and top=top_k.
    # Return a list of dicts with the relevant fields and the search score.
    # pass
    results = search_client.search(
        search_text=query,
        top=top_k,
        select=["id", "title", "problem_description", "solution", "category", "severity"]
    )

    return [
        {
            "id": r["id"],
            "title": r["title"],
            "problem_description": r["problem_description"],
            "solution": r["solution"],
            "category": r["category"],
            "severity": r["severity"],
            "@search.score": r["@search.score"]
        }
        for r in results
    ]


def semantic_search(query: str, top_k: int = 5) -> list[dict]:
    """Semantic search using the configuration you defined in Task 3."""
    # TODO 5.2: call search_client.search(...) with:
    #   query_type=QueryType.SEMANTIC,
    #   semantic_configuration_name="default-semantic-config",
    #   top=top_k,
    #   query_caption=QueryCaptionType.EXTRACTIVE,
    #   query_answer=QueryAnswerType.EXTRACTIVE,
    # Return dicts including both '@search.score' and '@search.reranker_score'.
    # pass
    results = search_client.search(
        search_text=query,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic-config",
        top=top_k,
        query_caption = QueryCaptionType.EXTRACTIVE,
        query_answer = QueryAnswerType.EXTRACTIVE,
        select=["id", "title", "problem_description", "solution", "category", "severity"]
    )
    return [
        {
            "id": r["id"],
            "title": r["title"],
            "problem_description": r["problem_description"],
            "solution": r["solution"],
            "category": r["category"],
            "severity": r["severity"],
            "@search.score": r["@search.score"],
            "@search.reranker_score": r.get("@search.reranker_score", 0) 
        }
        for r in results
    ]


def filtered_search(query: str, category: str, top_k: int = 5) -> list[dict]:
    """Semantic search restricted to a specific category."""
    # TODO 5.3: like semantic_search, but add filter=f"category eq '{category}'".
    # pass
    results = search_client.search(
        search_text=query,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic-config",
        top=top_k,
        query_caption = QueryCaptionType.EXTRACTIVE,
        query_answer = QueryAnswerType.EXTRACTIVE,
        filter=f"category eq '{category}'"
        select=["id", "title", "problem_description", "solution", "category", "severity"]
    )
    return [
        {
            "id": r["id"],
            "title": r["title"],
            "problem_description": r["problem_description"],
            "solution": r["solution"],
            "category": r["category"],
            "severity": r["severity"],
            "@search.score": r["@search.score"],
            "@search.reranker_score": r.get("@search.reranker_score", 0) 
        }
        for r in results
    ]

### Quick sanity check

After you implement the three functions, the cell below should print recognizable results from the service manual.

In [ ]:
# Run after the three retrieval functions are implemented
test_query = "my car will not start in the morning"

print("=== KEYWORD ===")
for r in keyword_search(test_query, top_k=3) or []:
    print(f"  [{r.get('@search.score', 0):.2f}] {r['title']}")

print("\n=== SEMANTIC ===")
for r in semantic_search(test_query, top_k=3) or []:
    print(f"  [rerank={r.get('@search.reranker_score', 0):.2f}] {r['title']}")

print("\n=== FILTERED (category=battery) ===")
for r in filtered_search(test_query, "battery", top_k=3) or []:
    print(f"  [rerank={r.get('@search.reranker_score', 0):.2f}] {r['title']}")

## Task 6 — Build the multi-turn RAG chat

**Your task:** implement a `ServiceChatBot` class that:

- maintains a conversation history (list of `{role, content}` messages)
- on each user turn:
  1. retrieves the top-k most relevant docs using **semantic_search** (default) or **filtered_search** if a category is supplied
  2. builds a system prompt that includes the retrieved evidence **and** instructions to:
     - answer only from the evidence
     - cite article ids in square brackets, e.g. `[svc-002]`
     - say "I don't have that in the service manual" when the evidence does not cover the question
     - keep technical tone but explain to a service advisor, not the end customer
  3. sends `[system_with_grounding] + history + new_user_message` to Azure OpenAI
  4. appends the assistant's reply to history and returns it

Public API the grader will call:

```python
bot = ServiceChatBot()
bot.ask("My Pajero Sport overheats in traffic", category=None)   # -> str
bot.ask("Could it also be the thermostat?")                       # follow-up turn
bot.reset()
```

**Hints:**
- The system prompt changes every turn because grounding changes — that's expected
- Keep the user/assistant history as plain text turns (no grounding inside history) so context stays clean
- Set `temperature` low (0.1-0.3) for factual grounding

In [ ]:
class ServiceChatBot:
    def __init__(self, deployment: str = AZURE_OPENAI_DEPLOYMENT, top_k: int = 4):
        self.deployment = deployment
        self.top_k = top_k
        self.history: list[dict] = []  # list of {"role": "user"|"assistant", "content": str}

    # TODO 6.1: implement reset() to clear self.history.
    def reset(self):
        self.history = []
        print("conversation cleared")

    # TODO 6.2: implement _format_grounding(docs) -> str.
    # It should turn a list of retrieved docs into a single string the LLM can read,
    # e.g. blocks of:
    #   [svc-002] Battery does not hold charge overnight
    #   Problem: ...
    #   Solution: ...
    def _format_grounding(self, docs: list[dict]) -> str:
        if not docs:
            return "No relevant articles in the service manual"
        blocks = []
        for doc in docs:
            block = (
                f"[{doc['id']}] {doc['title']}\n"
                f"Category: {doc['category']} | Severity: {doc['severity']}\n"
                f"Solution: {doc['solution']}"
            )
            blocks.append(block)
        return "\n\n".join(blocks)

    # TODO 6.3: implement _build_system_prompt(grounding: str) -> str using the
    # behavior rules listed in the markdown above.
    def _build_system_prompt(self, grounding: str) -> str:
        return (
            "You are a technical assistant for Mitsubishi Indonesia After-Sales service advisors. "
            "You help advisors accurately diagnose and resolve vehicle issues using the service manual knowledge base below.\n\n"
            "INSTRUCTIONS:\n"
            "- Answer ONLY using information from the SERVICE MANUAL EVIDENCE provided below.\n"
            "- Cite the article ID in square brackets after each relevant claim, e.g. [svc-002].\n"
            "- If the evidence does not cover the question, reply exactly: "
            "\"I don't have that in the service manual.\"\n"
            "- Keep a technical tone suitable for a trained service advisor, not the end customer.\n"
            "- Be concise and structured. Use numbered steps where applicable.\n\n"
            "SERVICE MANUAL EVIDENCE:\n"
            "---\n"
            f"{grounding}\n"
            "---"
        )

    # TODO 6.4: implement ask(user_message: str, category: str | None = None) -> str.
    # Steps:
    #   1) retrieve docs (semantic_search or filtered_search)
    #   2) build grounding + system prompt
    #   3) build messages = [system] + self.history + [{"role": "user", "content": user_message}]
    #   4) call aoai_client.chat.completions.create(...)
    #   5) append the user message AND the assistant reply to self.history
    #   6) return the assistant reply
    # pass
    def ask(self, user_message: str, category: str | None = None) -> str:
        if category:
            docs = filtered_search(user_message, category, top_k = self.top_k)
        else:
            docs = semantic_search(user_message, top_k=self.top_k)
        
        grounding = self._format_grounding(docs)
        system_prompt = self._build_system_prompt(grounding)

        messages = (
            [{"role": "system", "context": system_prompt}]
            + self.history
            + [{"role": "user", "content": user_message}]
        )

        response = aoai_client.chat.completions.create(
            model=self.deployment,
            messages=messages,
            temperature=0.2,
            max_tokens=600
        )
        reply = response.choices[0].message.content.strip()

        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": reply})

        return reply

### Smoke test the chatbot

After implementing the class, run a quick multi-turn conversation and verify:

- the second turn refers back to context from the first turn correctly
- citations like `[svc-002]` appear in the answer
- an out-of-scope question gets refused gracefully

In [ ]:
bot = ServiceChatBot()
print(bot.ask("My Pajero Sport overheats in traffic but is fine on the highway."))
print("---")
print(bot.ask("Could it also be the thermostat?"))
print("---")
print(bot.ask("What is the warranty period for a new battery?"))


## Task 7 — Evaluation: keyword vs semantic

Run the **same five questions** through `keyword_search` and `semantic_search` and compare the top-1 result. Then write a short analysis cell (markdown) describing what you observed.

**Required questions to run:**

```
1. "vehicle does not accelerate even when I press the gas"
2. "white smoke and steam from under the hood"
3. "ABS dashboard light won't turn off"
4. "battery dies after one night of parking"
5. "how often should I change my engine oil"
```

For each question, print:
- the top-1 keyword result (title + score)
- the top-1 semantic result (title + reranker score)
- whether they agree

**Then in a markdown cell below, answer in 4-6 sentences:**

- For which questions did semantic ranking change the top result, and was the change an improvement?
- Which question was best answered by both modes? Why?
- What does this tell you about when **schema design vs semantic ranking** matters most?

In [ ]:
eval_questions = [
    "vehicle does not accelerate even when I press the gas",
    "white smoke and steam from under the hood",
    "ABS dashboard light won't turn off",
    "battery dies after one night of parking",
    "how often should I change my engine oil",
]

# TODO 7.1: for each question, print the top-1 keyword result and the top-1 semantic result.
# Format suggestion:
# Q: <question>
#   keyword : [score] <title>
#   semantic: [rerank] <title>
#   agree   : True/False

print(f"{'Q#':<4} {'Question':<45} {'Mode':<10} {'Score':<8} {'Top-1 Title':<45} {'Agree?'}")
print("-" * 130)

for i, question in enumerate(eval_questions, 1):
    kw_results = keyword_search(question, top_k=1)
    sem_results = semantic_search(question, top_k=1)

    kw_top = kw_results[0] if kw_results else {"title": "N/A", "@search.score": 0}
    sem_top = sem_results[0] if sem_results else {"title": "N/A", "@search.reranker_score": 0}

    agree = kw_top["title"] == sem_top["title"]
    short_q = question[:43] + ".." if len(question) > 43 else question

    print(f"Q{i:<3} {short_q:<45} {'keyword':<10} {kw_top['@search.score']:<8.2f} {kw_top['title']:<45} {'✅' if agree else ''}")
    print(f"{'':4} {'':45} {'semantic':<10} {sem_top.get('@search.reranker_score', 0):<8.2f} {sem_top['title']:<45} {'✅' if agree else '❌ different'}")
    print()


### ✍️ Your analysis

_Write your 4-6 sentence analysis here. Replace this placeholder._

- **Where did semantic ranking change the result?**
- **Where did both agree?**
- **When does schema design matter more than semantic ranking?**

## ✅ Submission Checklist

Before submitting, verify:

- [ ] All API keys are loaded from **Colab Secrets**, not hardcoded
- [ ] The index `mitsubishi-service-manual` was created with **8 fields** and a **semantic configuration**
- [ ] All 15 documents uploaded successfully
- [ ] `keyword_search`, `semantic_search`, and `filtered_search` all return reasonable results
- [ ] `ServiceChatBot` correctly handles multi-turn conversation, citations, and out-of-scope questions
- [ ] Task 7 produces a side-by-side table for all 5 questions
- [ ] Task 7 analysis cell is written in your own words

## 🌟 Bonus (optional, no grade penalty if skipped)

- Add **hybrid search** (combine keyword and vector search) — you would need to add an embedding field and call Azure OpenAI's text-embedding deployment
- Implement **citation post-processing**: parse `[svc-XXX]` from the answer and attach the article title beside each citation
- Add a **Gradio** UI so a service advisor could try the bot in a browser

Good luck!